In [1]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from pathlib import Path
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Detect project root
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Project directories
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
HISTORY_DIR = OUTPUT_DIR / "training_history"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
HISTORY_DIR.mkdir(parents=True, exist_ok=True)

# Model configuration
IMG_HEIGHT = 64
IMG_WIDTH = 64
IMG_CHANNELS = 3
NUM_CLASSES = 43
BATCH_SIZE = 64

AUTOTUNE = tf.data.AUTOTUNE

print("TensorFlow version:", tf.__version__)
print("Project root:", PROJECT_ROOT)
print("Models directory:", MODELS_DIR)
print("Training history directory:", HISTORY_DIR)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.21.0
Project root: c:\Projects\Traffic_Sign_Recognition
Models directory: c:\Projects\Traffic_Sign_Recognition\models
Training history directory: c:\Projects\Traffic_Sign_Recognition\outputs\training_history
Available GPUs: []


In [2]:
train_df = pd.read_csv(PROCESSED_DIR / "train_metadata.csv")
val_df = pd.read_csv(PROCESSED_DIR / "validation_metadata.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test_metadata.csv")

print("TRAIN:", train_df.shape)
print("VALIDATION:", val_df.shape)
print("TEST:", test_df.shape)

print("\nNumber of classes:")
print("Train:", train_df["ClassId"].nunique())
print("Validation:", val_df["ClassId"].nunique())
print("Test:", test_df["ClassId"].nunique())

assert len(train_df) == 31367
assert len(val_df) == 7842
assert len(test_df) == 12630

assert train_df["ClassId"].nunique() == NUM_CLASSES
assert val_df["ClassId"].nunique() == NUM_CLASSES
assert test_df["ClassId"].nunique() == NUM_CLASSES

print("\nDataset metadata loaded and verified!")

TRAIN: (31367, 9)
VALIDATION: (7842, 9)
TEST: (12630, 9)

Number of classes:
Train: 43
Validation: 43
Test: 43

Dataset metadata loaded and verified!


In [3]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from pathlib import Path
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [4]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
HISTORY_DIR = PROJECT_ROOT / "outputs" / "training_history"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
HISTORY_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_HEIGHT = 64
IMG_WIDTH = 64
IMG_CHANNELS = 3
NUM_CLASSES = 43
BATCH_SIZE = 64

print("Project root:", PROJECT_ROOT)
print("Models directory:", MODELS_DIR)
print("Training history directory:", HISTORY_DIR)

Project root: c:\Projects\Traffic_Sign_Recognition
Models directory: c:\Projects\Traffic_Sign_Recognition\models
Training history directory: c:\Projects\Traffic_Sign_Recognition\outputs\training_history


In [5]:
train_df = pd.read_csv(PROCESSED_DIR / "train_metadata.csv")
val_df = pd.read_csv(PROCESSED_DIR / "validation_metadata.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test_metadata.csv")

print("TRAIN:", train_df.shape)
print("VALIDATION:", val_df.shape)
print("TEST:", test_df.shape)

print("\nNumber of classes:")
print("Train:", train_df["ClassId"].nunique())
print("Validation:", val_df["ClassId"].nunique())
print("Test:", test_df["ClassId"].nunique())

assert train_df["ClassId"].nunique() == NUM_CLASSES
assert val_df["ClassId"].nunique() == NUM_CLASSES
assert test_df["ClassId"].nunique() == NUM_CLASSES

print("\nDataset metadata loaded and verified!")

TRAIN: (31367, 9)
VALIDATION: (7842, 9)
TEST: (12630, 9)

Number of classes:
Train: 43
Validation: 43
Test: 43

Dataset metadata loaded and verified!


In [6]:
AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess_image(image_path, label):

    image_bytes = tf.io.read_file(image_path)

    image = tf.io.decode_png(
        image_bytes,
        channels=3
    )

    image = tf.image.convert_image_dtype(
        image,
        tf.float32
    )

    image = tf.image.resize(
        image,
        [IMG_HEIGHT, IMG_WIDTH]
    )

    label = tf.one_hot(
        label,
        depth=NUM_CLASSES
    )

    return image, label

In [7]:
def create_tf_dataset(
    dataframe,
    batch_size=BATCH_SIZE,
    training=False
):

    image_paths = dataframe["ImagePath"].astype(str).values

    labels = dataframe["ClassId"].astype(np.int32).values

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, labels)
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=min(len(dataframe), 10000),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_and_preprocess_image,
        num_parallel_calls=AUTOTUNE
    )

    dataset = dataset.batch(batch_size)

    dataset = dataset.prefetch(AUTOTUNE)

    return dataset

In [8]:
standard_augmentation = tf.keras.Sequential([
    layers.RandomRotation(
        factor=0.04,
        fill_mode="nearest",
        seed=SEED
    ),
    layers.RandomTranslation(
        height_factor=0.05,
        width_factor=0.05,
        fill_mode="nearest",
        seed=SEED
    ),
    layers.RandomZoom(
        height_factor=0.08,
        width_factor=0.08,
        fill_mode="nearest",
        seed=SEED
    ),
    layers.RandomContrast(
        factor=0.10,
        seed=SEED
    ),
    layers.RandomBrightness(
        factor=0.10,
        value_range=(0, 1),
        seed=SEED
    )
], name="standard_augmentation")


heavy_augmentation = tf.keras.Sequential([
    layers.RandomRotation(
        factor=0.08,
        fill_mode="nearest",
        seed=SEED
    ),
    layers.RandomTranslation(
        height_factor=0.10,
        width_factor=0.10,
        fill_mode="nearest",
        seed=SEED
    ),
    layers.RandomZoom(
        height_factor=0.15,
        width_factor=0.15,
        fill_mode="nearest",
        seed=SEED
    ),
    layers.RandomContrast(
        factor=0.25,
        seed=SEED
    ),
    layers.RandomBrightness(
        factor=0.20,
        value_range=(0, 1),
        seed=SEED
    )
], name="heavy_augmentation")


print("Standard and heavy augmentation pipelines created.")

Standard and heavy augmentation pipelines created.


In [9]:
def apply_augmentation(images, labels, augmentation_layer):

    images = augmentation_layer(
        images,
        training=True
    )

    images = tf.clip_by_value(
        images,
        0.0,
        1.0
    )

    return images, labels


def create_augmented_dataset(
    dataframe,
    augmentation_layer,
    batch_size=BATCH_SIZE
):

    dataset = create_tf_dataset(
        dataframe,
        batch_size=batch_size,
        training=True
    )

    dataset = dataset.map(
        lambda images, labels: apply_augmentation(
            images,
            labels,
            augmentation_layer
        ),
        num_parallel_calls=AUTOTUNE
    )

    return dataset.prefetch(AUTOTUNE)


train_dataset_standard = create_augmented_dataset(
    train_df,
    standard_augmentation
)

train_dataset_heavy = create_augmented_dataset(
    train_df,
    heavy_augmentation
)

val_dataset = create_tf_dataset(
    val_df,
    training=False
)

test_dataset = create_tf_dataset(
    test_df,
    training=False
)

print("All training and evaluation datasets created.")

All training and evaluation datasets created.


In [10]:
images, labels = next(iter(train_dataset_standard))

print("STANDARD TRAINING DATASET")
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Pixel range:", float(tf.reduce_min(images)), "to", float(tf.reduce_max(images)))

images, labels = next(iter(train_dataset_heavy))

print("\nHEAVY TRAINING DATASET")
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

images, labels = next(iter(val_dataset))

print("\nVALIDATION DATASET")
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

STANDARD TRAINING DATASET
Image batch shape: (64, 64, 64, 3)
Label batch shape: (64, 43)
Pixel range: 0.0 to 1.0

HEAVY TRAINING DATASET
Image batch shape: (64, 64, 64, 3)
Label batch shape: (64, 43)

VALIDATION DATASET
Image batch shape: (64, 64, 64, 3)
Label batch shape: (64, 43)


In [11]:
def build_baseline_cnn():

    model = models.Sequential([
        layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)),

        # Convolutional Block 1
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.20),

        # Convolutional Block 2
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Convolutional Block 3
        layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.30),

        # Classification Head
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.40),

        layers.Dense(NUM_CLASSES, activation="softmax")
    ], name="baseline_cnn")

    return model


baseline_model = build_baseline_cnn()

In [12]:
baseline_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Baseline CNN compiled successfully!")

Baseline CNN compiled successfully!


In [13]:
baseline_model.summary()

Model: "baseline_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 16, 16, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 43)             │        11,051 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,397,387 (9.15 MB)

 Trainable params: 2,396,427 (9.14 MB)

 Non-trainable params: 960 (3.75 KB)

In [14]:
total_params = baseline_model.count_params()

print("Model name:", baseline_model.name)
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{sum(np.prod(v.shape) for v in baseline_model.trainable_weights):,}")

Model name: baseline_cnn
Total parameters: 2,397,387
Trainable parameters: 2,396,427


In [15]:
from tensorflow.keras.utils import plot_model

architecture_path = MODELS_DIR / "baseline_cnn_architecture.png"

try:
    plot_model(
        baseline_model,
        to_file=str(architecture_path),
        show_shapes=True,
        show_layer_names=True,
        expand_nested=True,
        dpi=100
    )

    print("Architecture diagram saved to:", architecture_path)

except Exception as e:
    print("Architecture diagram could not be generated:", e)
    print("The model itself is still available.")

You must install pydot (`pip install pydot`) for `plot_model` to work.
Architecture diagram saved to: c:\Projects\Traffic_Sign_Recognition\models\baseline_cnn_architecture.png
